# Task 1 — Define Source and Extract

# Pluralsight Content Extraction

## Objective

This notebook extracts educational article data from Pluralsight's public AI & Data and Cloud blog categories.

The extracted data will be saved as raw JSON files and used as the input for Task 2 — Data Discovery, Profiling and Cleaning.

### Data Sources

1. Pluralsight AI & Data
   https://www.pluralsight.com/resources/blog/ai-and-data

2. Pluralsight Cloud
   https://www.pluralsight.com/resources/blog/cloud

The extracted metadata includes:

- Source
- Category
- Title
- Author
- Publication date
- Description
- Tags
- URL
- Article content
- Scraped timestamp

## Source Definition

### Source
Pluralsight public technology blog.

### Authentication
No authentication or API key is required for the public blog pages.

### Collection Method
The project uses Python HTTP requests and BeautifulSoup to retrieve and parse publicly accessible HTML pages.

### Categories
- AI & Data
- Cloud

### Pagination
The category pages are paginated. The scraper follows category pages and extracts individual article URLs.

### Rate Limiting
A delay of 2 seconds is applied between requests to reduce request frequency.

HTTP 429 responses are handled by waiting before retrying.

### Terms and Usage
The project is an educational data engineering project. Only publicly available article content and metadata are collected, and the scraper uses a low request rate.

In [37]:
from datetime import datetime
import json
import os
import time
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import requests

In [38]:
BASE_URL = "https://pluralsight.com"

# Tracks both targets and outputs to their explicit folder paths
CATEGORIES = {
    "AI & Data": {
        "url": "https://pluralsight.com/resources/blog/ai-and-data",
        "output": "../data/raw/pluralsight_ai_data_articles.json",
        "max_pages": 31,
    },
    "Cloud": {
        "url": "https://pluralsight.com/resources/blog/cloud",
        "output": "../data/raw/pluralsight_cloud_articles.json",
        "max_pages": 47,
    },
}

# Change this to 360 when you are ready to do the full historical crawl!
MAX_ARTICLES = 10
REQUEST_DELAY = 2
USER_AGENT = "learningProjectPipeline/1.0"

HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "en-US,en;q=0.9"}

session = requests.Session()
session.headers.update(HEADERS)

## HTTP Request Function

The following function retrieves a webpage and handles common HTTP errors.

A delay is applied after each request to reduce the request rate.

In [39]:
def fetch_page(url):
    """Download a webpage and return its HTML with rate-limit protections."""
    try:
        response = session.get(url, timeout=30)
        print(f"GET {url} -> {response.status_code}")

        if response.status_code == 200:
            time.sleep(REQUEST_DELAY)
            return response.text
        elif response.status_code == 429:
            print("Rate limited. Waiting 10 seconds...")
            time.sleep(10)
            return None
        return None
    except requests.RequestException as e:
        print(f"Request error: {e}")
        return None

## URL Normalization

Article URLs are normalized by removing query parameters and fragments.

This prevents the same article from being stored multiple times when it appears with different tracking parameters.

In [40]:
def normalize_url(url):
    """Clean query strings to prevent tracking duplicates."""
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path.rstrip('/')}"

## Article URL Discovery

The category pages are scanned to identify individual article URLs.

Only URLs belonging to the selected Pluralsight category are retained.

In [41]:
def extract_article_links(html, category_url):
    """Isolate article links that sit strictly under the current category tree structure."""
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    category_path = urlparse(category_url).path.rstrip("/") + "/"

    for a in soup.find_all("a", href=True):
        url = urljoin(BASE_URL, a["href"])
        url = normalize_url(url)
        parsed_path = urlparse(url).path

        if parsed_path.startswith(category_path):
            links.add(url)
    return links

In [42]:
def discover_article_urls(category_url, max_pages, max_articles):
    """Scrapes paginated list headers to fetch required candidate links."""
    article_urls = set()

    for page in range(1, max_pages + 1):
        if len(article_urls) >= max_articles:
            break

        page_url = (
            category_url if page == 1 else f"{category_url}?page={page}"
        )
        print(f"Scanning list page {page}...")

        html = fetch_page(page_url)
        if not html:
            continue

        links = extract_article_links(html, category_url)
        article_urls.update(links)

    return sorted(article_urls)[:max_articles]

## Article Metadata Extraction

Each article page is parsed to extract standardized metadata.

The extraction uses HTML selectors specific to the Pluralsight article structure.

The Table of Contents, navigation elements, scripts, styles, promotional content, and author biography are removed before article content is extracted.

In [43]:
def extract_article(url, html, category):
    """Parses individual article structural trees thoroughly across flexible layouts."""
    soup = BeautifulSoup(html, "html.parser")

    # --------------------------------------------------
    # REMOVE UNWANTED LOUT NODES UPFRONT
    # --------------------------------------------------
    toc = soup.find("div", class_="table-of-contents")
    if toc:
        toc.decompose()

    for el in soup.find_all(
        ["nav", "header", "footer", "aside", "script", "style"]
    ):
        el.decompose()

    # --------------------------------------------------
    # CORE METADATA FIELDS PARSING
    # --------------------------------------------------
    title = ""
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(" ", strip=True)

    description = ""
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc:
        description = meta_desc.get("content", "").strip()

    tags = []
    tag_list = soup.find("ul", class_="tag-list-listing")
    if tag_list:
        tags = [
            li.get_text(" ", strip=True)
            for li in tag_list.find_all("li")
            if li.get_text(strip=True)
        ]

    author = ""
    meta_auth = soup.find("meta", attrs={"name": "author"})
    if meta_auth:
        author = meta_auth.get("content", "").strip()

    if not author:
        for element in soup.find_all(["p", "div", "span"]):
            text = element.get_text(" ", strip=True)
            if text.startswith("By ") and len(text) < 100:
                author = text[3:].strip()
                break

    publication_date = ""
    date_el = soup.find("p", class_="date-length-text")
    if date_el:
        text = date_el.get_text(" ", strip=True)
        publication_date = text.split("•")[0].strip()

    # --------------------------------------------------
    # CHRONOLOGICAL CONTENT EXTRACTION ENGINE
    # --------------------------------------------------
    content_parts = []

    # Fallback cascade: Search inside target container blocks or scan the whole body
    content_area = soup.find("article") or soup.find("main") or soup.body

    if content_area:
        for element in content_area.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
            text = element.get_text(" ", strip=True)
            if not text:
                continue

            # Skip header paths, author bylines, or reading time stamps
            if text.startswith("By ") or "Minute Read" in text or "Read article" in text:
                continue
            if "Table of Contents" in text or "Copyright ©" in text:
                continue
            if "Terms of Use" in text or "Privacy Policy" in text:
                continue

            # Skip footer tags lists if found inline
            if element.name == "li":
                parent_classes = " ".join(
                    element.parent.get("class", [])
                ).lower()
                if "tag-list-listing" in parent_classes:
                    continue

            # Clean out promotional blocks text cleanly
            if "Advance your tech skills today" in text or "is a seasoned" in text:
                continue
            if "Subscribe to the newsletter" in text or "Free individual trial" in text:
                continue

            # Append text parts based on tag types
            if element.name in ["h1", "h2", "h3", "h4"]:
                content_parts.append(f"\n{text}\n")
            elif element.name == "li":
                content_parts.append(f"- {text}")
            else:
                content_parts.append(text)

    content = "\n\n".join(content_parts).strip()

    return {
        "source": "Pluralsight",
        "category": category,
        "title": title,
        "author": author,
        "publication_date": publication_date,
        "description": description,
        "tags": tags,
        "url": url,
        "content": content,
        "scraped_at": datetime.now().isoformat(),
    }


In [44]:
def load_existing_results(output_file):
    if os.path.exists(output_file):
        try:
            with open(output_file, "r", encoding="utf-8") as f:
                return json.load(f)
        except json.JSONDecodeError:
            return []
    return []


def save_results(results, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def scrape_category(category_name, category_config):
    output_file = category_config["output"]

    print("=" * 60)
    print(f"SCRAPING CATEGORY: {category_name}")
    print("=" * 60)

    existing_results = load_existing_results(output_file)
    existing_urls = {
        item["url"] for item in existing_results if item.get("url")
    }

    print(f"Existing stored records: {len(existing_results)}")

    article_urls = discover_article_urls(
        category_config["url"], category_config["max_pages"], MAX_ARTICLES
    )
    print(f"URLs selected for processing: {len(article_urls)}")

    results = existing_results.copy()

    for i, url in enumerate(article_urls, start=1):
        if url in existing_urls:
            print(f"[{i}/{len(article_urls)}] Already saved, skipping.")
            continue

        print(f"[{i}/{len(article_urls)}] Scraping: {url}")
        html = fetch_page(url)
        if not html:
            continue

        record = extract_article(url, html, category_name)
        if record and record["content"]:  # Ensures we don't save empty records
            results.append(record)
            save_results(results, output_file)
            print(f"Saved: {record['title']}")
        else:
            print("Warning: Article content parsing was empty. Not saving.")

    print(f"Total entries saved for {category_name}: {len(results)}\n")
    return results


In [45]:
# Iterates through both keys inside the configuration mapping dictionary seamlessly
all_results = {}

for category_name, category_config in CATEGORIES.items():
    results = scrape_category(category_name, category_config)
    all_results[category_name] = results

print("=" * 60)
print("ALL PIPELINE TARGET TASKS ARCHIVED SUCCESSFULLY")
print("=" * 60)

SCRAPING CATEGORY: AI & Data
Existing stored records: 0
Scanning list page 1...
GET https://pluralsight.com/resources/blog/ai-and-data -> 200
Scanning list page 2...
GET https://pluralsight.com/resources/blog/ai-and-data?page=2 -> 200
URLs selected for processing: 10
[1/10] Scraping: https://pluralsight.com/resources/blog/ai-and-data/10-emerging-ai-jobs
GET https://pluralsight.com/resources/blog/ai-and-data/10-emerging-ai-jobs -> 200
Saved: 10 emerging AI jobs to watch
[2/10] Scraping: https://pluralsight.com/resources/blog/ai-and-data/ai-accountability-agents
GET https://pluralsight.com/resources/blog/ai-and-data/ai-accountability-agents -> 200
Saved: AI accountability: Who's responsible when agents make bad calls?
[3/10] Scraping: https://pluralsight.com/resources/blog/ai-and-data/aws-certified-data-engineer-associate
GET https://pluralsight.com/resources/blog/ai-and-data/aws-certified-data-engineer-associate -> 200
Saved: AWS Certified Data Engineer - Associate (DEA-C01)
[4/10] Scra

## Extraction Results

The extracted records are stored as raw JSON files under `../data/raw/`.

The files produced by this notebook are:

- `../data/raw/pluralsight_ai_data_articles.json`
- `../data/raw/pluralsight_cloud_articles.json`

These files serve as the input to Task 2.